# ADRE-CXR — Kaggle Evidence Extraction

Run on Kaggle with **GPU enabled**. No training. No LLM needed here.

**Inputs to attach:**
1. `raddar/chest-xrays-indiana-university`
2. (Optional) Your `adre-cxr` code as a Kaggle dataset
3. (Optional) Guideline PDFs as `adre-cxr-guidelines`

In [ ]:
!pip install -q torchxrayvision torch torchvision sentence-transformers faiss-cpu rank-bm25 pymupdf pandas pyyaml tqdm scikit-image opencv-python-headless

import sys, json, shutil
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/chest-xrays-indiana-university")
WORK = Path("/kaggle/working/adre-cxr")
(WORK / "processed" / "evidence").mkdir(parents=True, exist_ok=True)

REPO = Path("/kaggle/input/adre-cxr-code") if Path("/kaggle/input/adre-cxr-code").exists() else Path("/kaggle/working")
if (REPO / "src").exists():
    shutil.copytree(REPO / "src", WORK / "src", dirs_exist_ok=True)
sys.path.insert(0, str(WORK))
print("Ready:", DATA_ROOT.exists())

In [ ]:
import pandas as pd
from src.data_prep.parse_reports import parse_reports_df
from src.data_prep.make_splits import make_splits, save_splits

reports = pd.read_csv(DATA_ROOT / "indiana_reports.csv")
parsed = parse_reports_df(reports)
dev, test = make_splits(parsed, dev_size=100, test_size=400, seed=42)
save_splits(dev, test, WORK / "processed")
print(f"Valid: {len(parsed)} | Dev: {len(dev)} | Test: {len(test)}")

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from src.evidence.evidence_packet import build_evidence_packet
from src.evidence.neural_inference import NeuralInference

projections = pd.read_csv(DATA_ROOT / "indiana_projections.csv")
neural = NeuralInference(weights="nih-rsna-14")
split = json.loads((WORK / "processed" / "dev_split.json").read_text())

for rec in tqdm(split, desc="Dev evidence"):
    uid = rec["uid"]
    proj = projections[projections["uid"] == int(uid)]
    if proj.empty:
        continue
    frontal = proj[proj["projection"] == "Frontal"] if "Frontal" in proj["projection"].values else proj
    row = frontal.iloc[0]
    img_path = DATA_ROOT / "images" / "images_normalized" / row["filename"]
    if not img_path.exists():
        continue
    packet = build_evidence_packet(
        study_id=str(uid), image_path=str(img_path),
        patient_metadata={"indication": rec.get("indication", "")},
        neural=neural,
    )
    packet.save(WORK / "processed" / "evidence" / f"{uid}.json")

In [ ]:
GUIDE_DIR = Path("/kaggle/input/adre-cxr-guidelines/guidelines")
if GUIDE_DIR.exists():
    from src.knowledge_index.parse_guidelines import parse_guideline_dir
    from src.knowledge_index.build_index import build_index
    chunks = parse_guideline_dir(GUIDE_DIR)
    build_index(chunks, WORK / "processed" / "index")
    print(f"Indexed {len(chunks)} chunks")
else:
    print("No guideline PDFs — attach optional dataset adre-cxr-guidelines")

## Download outputs

Save Version → download `/kaggle/working/adre-cxr/processed/` to your laptop's `data/processed/`.

Then run the LLM pipeline locally (Groq/Gemini — no GPU):

```bash
python scripts/run_pipeline.py --study-id 1 --image ... --evidence-json data/processed/evidence/1.json
```